# Day 10. Positional Encoding

**오늘의 목표**
- 문장에서 **순서**가 왜 중요한지, 그리고 Attention이 왜 순서를 모르는지 직접 확인한다
- GPT-2와 우리 mini GPT가 쓰는 **학습형 Position Embedding**을 만들고 `tok_emb + pos_emb`의 shape를 따라간다
- 원본 Transformer의 **Sinusoidal Positional Encoding** 공식을 구현하고 그래프로 본다
- 최신 LLM이 쓰는 **RoPE**의 핵심 아이디어(회전 → 상대 거리)를 2차원 예시로 이해한다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

## 1. 순서는 의미를 바꾼다

```text
개가 사람을 물었다    →  흔한 일
사람이 개를 물었다    →  뉴스감
```

쓰인 단어는 거의 같은데 **순서**만 바뀌어도 뜻이 완전히 달라집니다.
그러니 모델은 "어떤 Token이 있나"뿐 아니라 "**몇 번째에** 있나"도 알아야 해요.

## 2. 문제: Attention은 순서를 모른다

2주차에 배울 Self-Attention은 대략 이런 계산입니다. (Day 2, Day 5에서 맛본 그 계산)

```text
scores  = X @ X.T          # 모든 Token 쌍의 관련도
weights = softmax(scores)  # 행마다 확률로
out     = weights @ X      # 관련도만큼 섞기
```

이 계산에는 "몇 번째"라는 정보가 **어디에도 없습니다**. 직접 확인해 봅시다.

In [ ]:
tokens = ["나는", "오늘", "학교에", "간다"]
T, C = 4, 8
X = torch.randn(T, C)          # 각 Token의 Embedding이라고 생각하세요 (Day 9)

def order_blind_attention(X):
    weights = F.softmax(X @ X.T, dim=-1)
    return weights @ X

out = order_blind_attention(X)

# 순서를 섞기: 간다, 학교에, 나는, 오늘
perm = torch.tensor([3, 2, 0, 1])
X_shuffled = X[perm]
out_shuffled = order_blind_attention(X_shuffled)

print("섞은 순서:", [tokens[i] for i in perm])
print()
for i, p in enumerate(perm):
    same = torch.allclose(out_shuffled[i], out[p], atol=1e-6)
    print(f"섞은 문장의 '{tokens[p]}' 출력 == 원래 문장의 '{tokens[p]}' 출력 ? {same}")

순서를 뒤섞어도 **각 Token의 출력은 똑같고, 자리만 바뀌었을 뿐**입니다.
즉 모델 입장에서 `나는 오늘 학교에 간다`와 `간다 학교에 나는 오늘`은 **구별이 안 됩니다.**

> 💡 해결책: Token 벡터에 **"나는 몇 번째 자리다"라는 정보를 더해서** 넣어준다 → Positional Encoding

## 3. 단순한 아이디어들과 그 문제점

### 3-1. 위치 번호(0, 1, 2, ...)를 그대로 더하기

In [ ]:
x = torch.randn(C) * 0.5    # 보통 Embedding 값은 이 정도 크기
for pos in [0, 1, 10, 500]:
    print(f"위치 {pos:3d}: 원래 벡터 {r(x[:3], 2)}  →  더한 후 {r((x + pos)[:3], 2)}")

위치가 커질수록 더한 숫자가 **원래 의미(Embedding)를 덮어버립니다.** 500번째 Token은 원래 무슨 단어였는지 알아보기 어려워요.

### 3-2. 0~1로 정규화하기 (`t / T`)

크기 문제는 해결되지만, **문장 길이에 따라 같은 값의 의미가 달라집니다.**

In [ ]:
for T_len in [4, 10]:
    print(f"길이 {T_len:2d} 문장의 위치값:", [round(t / T_len, 2) for t in range(T_len)])

길이 4 문장에서 `0.2`에 가장 가까운 값은 두 번째 자리(0.25)지만, 길이 10 문장에서 `0.2`는 세 번째 자리입니다. 같은 값이 문장마다 다른 자리를 뜻하고, 이웃한 Token 사이 간격(0.25 vs 0.1)도 문장 길이에 따라 달라집니다. 모델이 일관된 규칙을 배우기 어려워요.

**그래서 실제로는 숫자 하나가 아니라, Embedding과 같은 길이 `C`의 위치 벡터를 만들어 더합니다.**
방법은 크게 세 가지예요.

| 방법 | 쓰는 모델 | 핵심 |
|---|---|---|
| 학습형 Position Embedding | GPT-2, **우리 mini GPT** | 위치마다 벡터를 학습 |
| Sinusoidal Encoding | 원본 Transformer (2017) | sin/cos 공식으로 고정 |
| RoPE | LLaMA, Qwen 등 최신 LLM | Q/K를 위치만큼 회전 |

## 4. 학습형 Position Embedding (GPT-2, mini GPT)

가장 간단합니다. **Token Embedding과 똑같은 방식**으로 위치 Embedding 표를 하나 더 만듭니다.

```text
Token Embedding    : nn.Embedding(vocab_size, C)   → "무슨 Token인가"
Position Embedding : nn.Embedding(block_size, C)   → "몇 번째인가"
```

`block_size`는 모델이 한 번에 볼 수 있는 **최대 Token 개수**입니다.

In [ ]:
vocab_size, block_size, C = 65, 8, 16
B, T = 2, 4

token_embedding    = nn.Embedding(vocab_size, C)
position_embedding = nn.Embedding(block_size, C)

idx = torch.randint(0, vocab_size, (B, T))     # Token ID들 (Day 8 Tokenizer 결과)
pos = torch.arange(T)                          # 0, 1, 2, 3

tok_emb = token_embedding(idx)                 # (B, T, C)
pos_emb = position_embedding(pos)              # (T, C)
x = tok_emb + pos_emb                          # (B, T, C) + (T, C) → ?

print("idx     :", tuple(idx.shape))
print("pos     :", pos.tolist())
print("tok_emb :", tuple(tok_emb.shape))
print("pos_emb :", tuple(pos_emb.shape))
print("x       :", tuple(x.shape))

### 4-1. `(B, T, C) + (T, C)`는 어떻게 더해지나? — Broadcasting

shape가 달라도 PyTorch가 **오른쪽 축부터 맞춰서**, 없는 축은 복사해서 더해줍니다.

```text
tok_emb : (B, T, C) = (2, 4, 16)
pos_emb :    (T, C) =    (4, 16)   ← 앞에 B 축이 없으니 문장마다 똑같이 복사
결과    : (B, T, C) = (2, 4, 16)
```

즉 **모든 문장의 0번째 자리에는 같은 위치 벡터**, 1번째 자리에는 같은 위치 벡터… 가 더해집니다.

In [ ]:
manual = tok_emb + pos_emb.unsqueeze(0).expand(B, T, C)   # B 축을 직접 만들어 복사
print("broadcasting 결과 == 직접 복사해서 더한 결과 ?", torch.allclose(x, manual))
print("두 문장의 0번째 자리에 더해진 위치 벡터가 같은가?",
      torch.allclose(x[0, 0] - tok_emb[0, 0], x[1, 0] - tok_emb[1, 0]))

### 4-2. 이제 순서가 구별될까?

2절의 실험을 위치 벡터를 더해서 다시 해봅시다.

In [ ]:
T, C = 4, 8
torch.manual_seed(0)
X = torch.randn(T, C)
P = torch.randn(T, C)                   # 위치 0~3의 위치 벡터

out          = order_blind_attention(X + P)
out_shuffled = order_blind_attention(X[perm] + P)   # Token은 섞였지만 위치 벡터는 0,1,2,3 그대로

for i, p in enumerate(perm):
    same = torch.allclose(out_shuffled[i], out[p], atol=1e-6)
    print(f"'{tokens[p]}' 출력이 같은가? {same}")

이제 같은 Token이라도 **자리가 다르면 다른 출력**이 나옵니다. 모델이 순서를 구별할 수 있게 됐어요.

### 4-3. 한계: `block_size`를 넘을 수 없다

In [ ]:
try:
    position_embedding(torch.arange(block_size + 1))   # 위치 0~8, 표에는 0~7만 있음
except IndexError as e:
    print("IndexError:", e)

학습형 Position Embedding은 **표에 있는 위치까지만** 쓸 수 있습니다.
그래서 GPT-2는 1024 Token, mini GPT는 `block_size`까지만 입력을 받을 수 있어요.
텍스트를 생성할 때 입력이 길어지면 **최근 `block_size`개만 잘라서** 넣는 이유가 이거예요. (Day 28)

## 5. Sinusoidal Positional Encoding (원본 Transformer)

학습하지 않고 **공식으로** 위치 벡터를 만듭니다.

```text
PE(pos, 2i)   = sin(pos / 10000^(2i / C))
PE(pos, 2i+1) = cos(pos / 10000^(2i / C))
```

- `pos`: 위치 (0, 1, 2, ...)
- `i`: 차원 쌍 번호. 짝수 차원은 sin, 홀수 차원은 cos
- 앞쪽 차원은 **빠르게** 진동하고, 뒤쪽 차원은 **천천히** 진동합니다

시계 바늘에 비유하면: 초침(앞 차원) · 분침 · 시침(뒤 차원)을 모두 보면 정확한 시각을 알 수 있는 것처럼, 여러 속도의 파동을 모두 보면 정확한 위치를 알 수 있어요.

### 5-1. 구현하기

In [ ]:
def sinusoidal_encoding(T, C):
    pe = torch.zeros(T, C)
    pos = torch.arange(T).unsqueeze(1).float()          # (T, 1)
    i2  = torch.arange(0, C, 2).float()                 # 0, 2, 4, ... (= 2i)
    freq = 1.0 / (10000 ** (i2 / C))                    # (C/2,)
    pe[:, 0::2] = torch.sin(pos * freq)                 # 짝수 차원
    pe[:, 1::2] = torch.cos(pos * freq)                 # 홀수 차원
    return pe

pe = sinusoidal_encoding(T=50, C=64)
print("shape:", tuple(pe.shape))
print("위치 0:", r(pe[0, :6]))
print("위치 1:", r(pe[1, :6]))

위치 0은 `sin(0)=0`, `cos(0)=1`이 번갈아 나오는 `[0, 1, 0, 1, ...]`입니다.

### 5-2. 그래프로 보기

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

im = axes[0].imshow(pe.T, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
axes[0].set_xlabel("위치 (pos)")
axes[0].set_ylabel("차원")
axes[0].set_title("Sinusoidal Encoding (위치 50 × 차원 64)", fontsize=13)
fig.colorbar(im, ax=axes[0])

for d in [0, 4, 10, 20]:
    axes[1].plot(pe[:, d], linewidth=2, label=f"차원 {d}")
axes[1].set_xlabel("위치 (pos)")
axes[1].set_title("차원마다 진동 속도가 다르다", fontsize=13)
axes[1].legend(loc="lower left")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

- 왼쪽: 세로줄 하나가 위치 하나의 벡터입니다. **위쪽(앞 차원)은 촘촘하게, 아래쪽(뒤 차원)은 느리게** 바뀌어요.
- 왼쪽 그림의 가로 줄무늬는 짝수 차원(sin)과 홀수 차원(cos)이 번갈아 있어서 생깁니다. 뒤쪽 차원은 위치 50까지 거의 안 변해서 sin은 0 근처, cos는 1 근처로 보여요.
- 오른쪽: 차원 0은 빠르게, 차원 20은 아주 천천히 진동합니다. (정수 위치에서만 값을 찍어서 빠른 파동은 각져 보여요)

### 5-3. 가까운 위치끼리 비슷하다

Day 2에서 Dot Product = "얼마나 비슷한가"라고 배웠죠. 위치 벡터끼리 Dot Product를 해봅시다.

In [ ]:
sim = pe @ pe.T          # (50, 50): 모든 위치 쌍의 Dot Product

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
im = axes[0].imshow(sim, cmap="viridis")
axes[0].set_xlabel("위치 n")
axes[0].set_ylabel("위치 m")
axes[0].set_title("위치 벡터끼리의 Dot Product", fontsize=13)
fig.colorbar(im, ax=axes[0])

axes[1].plot(sim[20], linewidth=2.5)
axes[1].axvline(20, color="gray", linestyle="--")
axes[1].set_xlabel("위치 n")
axes[1].set_title("위치 20과 다른 위치들의 Dot Product", fontsize=13)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

대각선(자기 자신)이 가장 밝고, **멀어질수록 어두워집니다.**
위치 벡터만 봐도 "가까운 자리 / 먼 자리"가 구별된다는 뜻이에요.

| | 학습형 | Sinusoidal |
|---|---|---|
| 만드는 법 | 학습으로 찾음 | 공식으로 고정 |
| 파라미터 | `block_size × C`개 | 0개 |
| `block_size`보다 긴 입력 | 불가능 | 공식상 계산은 가능 |
| 쓰는 곳 | GPT-2, mini GPT | 원본 Transformer |

GPT-2 연구진은 학습형이 성능이 비슷하면서 더 단순해서 학습형을 골랐습니다.

## 6. RoPE (Rotary Position Embedding) — 개념만

LLaMA, Qwen, Mistral 등 요즘 LLM 대부분이 쓰는 방법입니다.

**아이디어**: 위치 벡터를 **더하는** 대신, Attention의 Q와 K 벡터를 **위치만큼 회전**시킵니다.

```text
위치 m의 Query  →  각도 m·θ 만큼 회전
위치 n의 Key    →  각도 n·θ 만큼 회전
두 벡터의 Dot Product → 각도 차이 (m - n)·θ 에만 영향을 받음
```

즉 "절대 위치"가 아니라 **"둘이 몇 칸 떨어져 있나(상대 거리)"** 가 자연스럽게 Attention 점수에 반영됩니다.

### 6-1. 2차원으로 직접 확인하기

In [ ]:
def rotate(v, angle):
    # 2차원 벡터 v를 angle(라디안)만큼 회전
    c, s = math.cos(angle), math.sin(angle)
    R = torch.tensor([[c, -s],
                      [s,  c]])
    return R @ v

theta = 0.3
q = torch.tensor([1.0, 0.5])     # 어떤 Token의 Query
k = torch.tensor([0.8, -0.2])    # 어떤 Token의 Key

def rope_score(q, k, m, n):
    return rotate(q, m * theta) @ rotate(k, n * theta)

for m, n in [(3, 1), (5, 3), (10, 8), (100, 98), (5, 1)]:
    print(f"Query 위치 {m:3d}, Key 위치 {n:3d}  (거리 {m - n})  →  점수 {rope_score(q, k, m, n).item():.4f}")

위치가 `(3,1)`, `(5,3)`, `(10,8)`, `(100,98)`로 전부 달라도 **거리가 2로 같으면 점수가 같습니다.**
거리가 4인 `(5,1)`만 다른 값이 나와요.

> 💡 실제 RoPE는 `C`차원 벡터를 2개씩 짝지어 쌍마다 다른 속도(θ)로 회전시킵니다. Sinusoidal의 "여러 속도의 파동" 아이디어와 같아요.
> 우리 mini GPT는 GPT-2처럼 학습형 Position Embedding을 씁니다. RoPE는 "최신 모델은 이렇게 한다" 정도로 알아두면 충분해요.

## 7. LLM과의 연결

커리큘럼 6-9의 최종 GPT 구조에서 오늘 배운 부분은 **맨 첫 단계**입니다.

```text
Token ID (B, T)
   ↓                        ↓
Token Embedding          Position Embedding(arange(T))
(B, T, C)                (T, C)
   ↓                        ↓
   └──────── + ─────────────┘
             ↓
        x (B, T, C)       ← 오늘 여기까지
             ↓
   Transformer Block × N   (Day 11~ Attention)
             ↓
         LayerNorm → Linear → Vocabulary Logits
```

실제 nanoGPT 코드도 거의 같습니다.

```python
tok_emb = self.token_embedding_table(idx)                    # (B, T, C)
pos_emb = self.position_embedding_table(torch.arange(T))     # (T, C)
x = tok_emb + pos_emb                                        # (B, T, C)
```

## 8. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** `vocab_size=100`, `block_size=16`, `C=32`인 Token/Position Embedding을 만들고,
`idx`를 넣어서 `x = tok_emb + pos_emb`를 계산하세요.

In [ ]:
vocab_size, block_size, C = 100, 16, 32
idx = torch.randint(0, vocab_size, (3, 10))     # (B=3, T=10)
B, T = idx.shape

token_embedding    = None  # TODO
position_embedding = None  # TODO

tok_emb = None  # TODO
pos_emb = None  # TODO: 힌트 torch.arange
x = tok_emb + pos_emb

assert tuple(pos_emb.shape) == (T, C), "pos_emb shape를 다시 확인하세요"
assert tuple(x.shape) == (B, T, C), "x shape를 다시 확인하세요"
assert torch.allclose(x[1, 4] - token_embedding(idx[1, 4]), position_embedding.weight[4]), "위치 번호가 맞나요?"
print("Q1 정답!", tuple(x.shape))

**Q2.** Broadcasting 결과 shape를 실행하기 **전에** 예측하세요. 더할 수 없으면 `None`이라고 적으세요.

In [ ]:
answers = {
    "(2, 5, 8) + (5, 8)": "?",   # TODO
    "(2, 5, 8) + (8,)":   "?",   # TODO
    "(2, 5, 8) + (6, 8)": "?",   # TODO
}

def try_add(a, b):
    try:
        return tuple((torch.zeros(*a) + torch.zeros(*b)).shape)
    except RuntimeError:
        return None

checks = {
    "(2, 5, 8) + (5, 8)": try_add((2, 5, 8), (5, 8)),
    "(2, 5, 8) + (8,)":   try_add((2, 5, 8), (8,)),
    "(2, 5, 8) + (6, 8)": try_add((2, 5, 8), (6, 8)),
}
for key in checks:
    mark = "✅" if answers[key] == checks[key] else "❌"
    print(f"{mark} {key:20s} 내 답: {answers[key]}")

**Q3.** Sinusoidal 공식을 이용해 **위치 `pos` 하나의 벡터**를 반복문으로 직접 만드세요. (`C`는 짝수)

```text
PE(pos, 2i)   = sin(pos / 10000^(2i / C))
PE(pos, 2i+1) = cos(pos / 10000^(2i / C))
```

In [ ]:
def pe_one_position(pos, C):
    v = torch.zeros(C)
    for i in range(C // 2):
        pass  # TODO: v[2*i]와 v[2*i+1]을 채우세요
    return v

assert torch.allclose(pe_one_position(7, 16), sinusoidal_encoding(20, 16)[7], atol=1e-6), "공식을 다시 확인하세요"
assert torch.allclose(pe_one_position(0, 8), sinusoidal_encoding(1, 8)[0], atol=1e-6)
print("Q3 정답!")

**Q4.** 6-1의 `rope_score`를 사용합니다. 아래 위치 쌍 중 **`(4, 1)`과 점수가 같은 쌍**을 모두 고르세요.

In [ ]:
candidates = [(7, 4), (4, 2), (13, 10), (1, 4), (30, 27)]

my_answer = "?"   # TODO: 리스트로. 예) [(7, 4)]

target = rope_score(q, k, 4, 1)
correct = [c for c in candidates if torch.allclose(rope_score(q, k, *c), target, atol=1e-5)]
assert isinstance(my_answer, list) and sorted(my_answer) == sorted(correct), "RoPE 점수는 무엇에만 영향을 받나요?"
print("Q4 정답!")

## 9. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Position 정보는 왜 필요한가? Attention만으로는 왜 순서를 알 수 없는가?
2. 위치 번호(0, 1, 2, ...)를 그대로 더하면 어떤 문제가 생기는가?
3. `tok_emb (B, T, C) + pos_emb (T, C)`는 어떻게 더해지는가?
4. 학습형 Position Embedding은 왜 `block_size`보다 긴 입력을 받을 수 없는가?
5. RoPE에서 두 Token의 Attention 점수는 무엇에 영향을 받는가?

**내 답:**

1.
2.
3.
4.
5.